In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from scipy.sparse import csr_matrix

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
path = "/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations/"

articles = pd.read_csv(
    path + "articles.csv",
    dtype={"article_id": str},
    usecols=[
        "article_id",
        "prod_name",
        "product_type_name",
        "product_group_name",
        "colour_group_name",
        "detail_desc"
    ]
)

customers = pd.read_csv(
    path + "customers.csv",
    usecols=[
        "customer_id",
        "age",
        "club_member_status"
    ]
)

transactions = pd.read_csv(
    path + "transactions_train.csv",
    dtype={"article_id": str},
    usecols=[
        "t_dat",
        "customer_id",
        "article_id",
        "price"
    ]
)

print("Articles:", articles.shape)
print("Customers:", customers.shape)
print("Transactions:", transactions.shape)

Articles: (105542, 6)
Customers: (1371980, 3)
Transactions: (31788324, 4)


In [4]:
#We're going to reduce noise 
# Convert date column
transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

# Keep only last 12 months
cutoff_date = transactions["t_dat"].max() - pd.DateOffset(months=12)

trans_filtered = transactions[
    transactions["t_dat"] >= cutoff_date
].copy()

print("Transactions after date filter:", len(trans_filtered))
print(
    "Date range:",
    trans_filtered["t_dat"].min().date(),
    "to",
    trans_filtered["t_dat"].max().date()
)

Transactions after date filter: 14932373
Date range: 2019-09-22 to 2020-09-22


In [5]:
# Keep active users (5+ purchases)

user_counts = trans_filtered.groupby(
    "customer_id"
)["article_id"].count()

active_users = user_counts[
    user_counts >= 5
].index

trans_filtered = trans_filtered[
    trans_filtered["customer_id"].isin(active_users)
]

print("Transactions:", len(trans_filtered))
print(
    "Unique users:",
    trans_filtered["customer_id"].nunique()
)

Transactions: 14121262
Unique users: 647022


In [6]:
# Keep active items (5+ purchases)

item_counts = trans_filtered.groupby(
    "article_id"
)["customer_id"].count()

active_items = item_counts[
    item_counts >= 5
].index

trans_filtered = trans_filtered[
    trans_filtered["article_id"].isin(active_items)
]

print("Transactions:", len(trans_filtered))
print(
    "Unique items:",
    trans_filtered["article_id"].nunique()
)

Transactions: 14087690
Unique items: 54075


In [7]:

transactions_sample = trans_filtered.sample(
    n=1_000_000,
    random_state=42
).copy()

print("Sample size:", len(transactions_sample))
print("Unique users:", transactions_sample["customer_id"].nunique())
print("Unique items:", transactions_sample["article_id"].nunique())

Sample size: 1000000
Unique users: 417324
Unique items: 45532


In [9]:
#train test split 
# Users with at least 2 purchases

user_counts = transactions_sample.groupby(
    "customer_id"
)["article_id"].count()

valid_users = user_counts[
    user_counts >= 2
].index

transactions_sample = transactions_sample[
    transactions_sample["customer_id"].isin(valid_users)
].copy()

print(
    "Remaining transactions:",
    len(transactions_sample)
)

print(
    "Remaining users:",
    transactions_sample["customer_id"].nunique()
)

Remaining transactions: 812212
Remaining users: 229536


In [10]:
transactions_sample = transactions_sample.sort_values("t_dat")

test_df = transactions_sample.groupby(
    "customer_id"
).tail(1)

train_df = transactions_sample.drop(
    test_df.index
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

print(
    "Train users:",
    train_df["customer_id"].nunique()
)

print(
    "Test users:",
    test_df["customer_id"].nunique()
)

Train size: 582676
Test size: 229536
Train users: 229536
Test users: 229536


In [11]:
#creating user item mappping
user_ids = train_df["customer_id"].unique()
item_ids = train_df["article_id"].unique()

user2idx = {
    uid: idx
    for idx, uid in enumerate(user_ids)
}

item2idx = {
    iid: idx
    for idx, iid in enumerate(item_ids)
}

idx2user = {
    idx: uid
    for uid, idx in user2idx.items()
}

idx2item = {
    idx: iid
    for iid, idx in item2idx.items()
}

print("Users:", len(user_ids))
print("Items:", len(item_ids))

Users: 229536
Items: 41194


In [12]:
train_df["user_idx"] = train_df[
    "customer_id"
].map(user2idx)

train_df["item_idx"] = train_df[
    "article_id"
].map(item2idx)

print(train_df.shape)

print(
    train_df[
        ["customer_id",
         "article_id",
         "user_idx",
         "item_idx"]
    ].head()
)

(582676, 6)
                                                customer_id  article_id  \
16882607  ca9420f8c4043cf108dfdf5d0d2cc874eec28a37ca6516...  0779863001   
16884253  d6c0450dba2cfb7064a63ba86ae97491825ae22669f48e...  0801309002   
16868262  5dd939de0280c005a6ee75f446926515f6757a93be7585...  0762846004   
16865518  48de9e94fd05b82f4a4c0e43e4f5e755345314bdc905eb...  0706016003   
16859136  1901e820f4023f5e6638f6d71de35d1a3c71ab68e0d05a...  0678942031   

          user_idx  item_idx  
16882607         0         0  
16884253         1         1  
16868262         2         2  
16865518         3         3  
16859136         4         4  


In [13]:
from scipy.sparse import csr_matrix

interaction = train_df.groupby(
    ["user_idx", "item_idx"]
).size().reset_index(name="count")

user_item_matrix = csr_matrix(
    (
        interaction["count"],
        (
            interaction["user_idx"],
            interaction["item_idx"]
        )
    ),
    shape=(
        len(user_ids),
        len(item_ids)
    )
)

print("Matrix shape:", user_item_matrix.shape)
print("Interactions:", user_item_matrix.nnz)

sparsity = (
    1 -
    user_item_matrix.nnz /
    (
        user_item_matrix.shape[0]
        *
        user_item_matrix.shape[1]
    )
)

print(f"Sparsity: {sparsity:.4%}")

Matrix shape: (229536, 41194)
Interactions: 574643
Sparsity: 99.9939%


In [14]:
!pip install implicit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 60.3 MB/s eta 0:00:0000:0100:01


In [56]:
import implicit

confidence = user_item_matrix.astype("double")

als_model = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.05,
    iterations=20,
    random_state=42
)

print("Training ALS...")
als_model.fit(confidence)

print("ALS training completed!")

Training ALS...


  0%|          | 0/20 [00:00<?, ?it/s]

ALS training completed!


In [57]:
print("User factors:", als_model.user_factors.shape)
print("Item factors:", als_model.item_factors.shape)

User factors: (229536, 64)
Item factors: (41194, 64)


In [17]:
# Keep only items present in training

article_subset = articles[
    articles["article_id"].isin(item_ids)
].copy()

print(article_subset.shape)

(41194, 6)


In [19]:
article_subset["detail_desc"] = article_subset["detail_desc"].fillna("")

article_subset["text"] = (
    article_subset["prod_name"].fillna("") + " " +
    article_subset["product_type_name"].fillna("") + " " +
    article_subset["product_group_name"].fillna("") + " " +
    article_subset["colour_group_name"].fillna("") + " " +
    article_subset["detail_desc"].fillna("")
)

article_subset["text"].head()

0    Strap top Vest top Garment Upper body Black Je...
1    Strap top Vest top Garment Upper body White Je...
3    OP T-shirt (Idro) Bra Underwear Black Microfib...
4    OP T-shirt (Idro) Bra Underwear White Microfib...
5    OP T-shirt (Idro) Bra Underwear Light Beige Mi...
Name: text, dtype: object

In [21]:
!pip install sentence-transformers -q

In [22]:
from sentence_transformers import SentenceTransformer

model_st = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully


In [23]:
texts = article_subset["text"].tolist()

print("Total products:", len(texts))

Total products: 41194


In [24]:
product_embeddings = model_st.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", product_embeddings.shape)

Batches:   0%|          | 0/161 [00:00<?, ?it/s]

Embeddings shape: (41194, 384)


In [25]:
import pickle

pickle.dump(
    product_embeddings,
    open("product_embeddings.pkl", "wb")
)

print("Embeddings saved successfully")

Embeddings saved successfully


In [26]:
article_to_embedding_idx = {
    article_id: idx
    for idx, article_id in enumerate(
        article_subset["article_id"]
    )
}

print(
    "Mapping size:",
    len(article_to_embedding_idx)
)

Mapping size: 41194


In [35]:
from sklearn.metrics.pairwise import cosine_similarity

def get_similar_products(article_id, top_n=10):

    if article_id not in article_to_embedding_idx:
        return []

    idx = article_to_embedding_idx[article_id]

    query_embedding = product_embeddings[idx].reshape(1, -1)

    similarities = cosine_similarity(
        query_embedding,
        product_embeddings
    )[0]

    similar_indices = similarities.argsort()[::-1]

    query_name = article_subset.iloc[idx]["prod_name"]

    seen = set([query_name])
    results = []

    for i in similar_indices:

        product_name = article_subset.iloc[i]["prod_name"]

        if product_name in seen:
            continue

        seen.add(product_name)
        results.append(product_name)

        if len(results) >= top_n:
            break

    return results

In [37]:
sample_article = article_subset.iloc[0]["article_id"]

print("Query Product:")
print(article_subset.iloc[0]["prod_name"])

recommendations = get_similar_products(
    sample_article,
    top_n=5
)

print("\nSimilar Products:")

for r in recommendations:
    print("-", r)

Query Product:
Strap top

Similar Products:
- V-neck strap top
- Strap top 3-pack
- Strap Top 2 pack
- Strap Top.
- Straptop 2-pack


In [59]:
# test als 

def get_als_recommendations(user_idx, n=10):

    ids, scores = als_model.recommend(
        userid=user_idx,
        user_items=user_item_matrix[user_idx],
        N=n,
        filter_already_liked_items=True
    )

    return ids, scores

In [49]:
#temporary test case 
ids, scores = als_model.recommend(
    userid=0,
    user_items=user_item_matrix[0],
    N=20,
    filter_already_liked_items=True
)

print(ids)
print("Max returned ID:", max(ids))

[ 10274  10954  53773 186603  19439  18860  89918  53303  60015  20976
 153121  30708 106747  86080  97512  29179 130088  12937  90717  25065]
Max returned ID: 186603


In [41]:
sample_user = 0

ids, scores = get_als_recommendations(
    sample_user,
    n=5
)

print("Recommended Item IDs:")
print(ids)

print("\nScores:")
print(scores)

Recommended Item IDs:
[ 10274  10954  53773 186603  19439]

Scores:
[0.00708484 0.00635198 0.00611577 0.00601414 0.00595813]


In [42]:
for item_id, score in zip(ids, scores):

    article_id = idx2item[item_id]

    name = article_subset[
        article_subset["article_id"] == article_id
    ]["prod_name"]

    if len(name) > 0:
        print(
            f"{name.iloc[0]} | Score: {score:.5f}"
        )

Bowtie Kalle | Score: 0.00708
Classic Boilersuit | Score: 0.00635


KeyError: np.int32(53773)

In [43]:
print("Number of items:", len(item_ids))
print("Number of users:", len(user_ids))

print("ALS user factors:", als_model.user_factors.shape)
print("ALS item factors:", als_model.item_factors.shape)

Number of items: 41194
Number of users: 229536
ALS user factors: (41194, 64)
ALS item factors: (229536, 64)


In [44]:
print("Max recommendation id:", max(ids))
print("Min recommendation id:", min(ids))

Max recommendation id: 186603
Min recommendation id: 10274


In [45]:
print(user_item_matrix.shape)
print(item_user_matrix.shape)

(229536, 41194)
(41194, 229536)


In [46]:
sample_item = 0

ids, scores = als_model.recommend(
    sample_item,
    item_user_matrix[sample_item],
    N=5
)

print(ids)
print(scores)

[ 10954  53773 186603  19439  18860]
[0.00635198 0.00611577 0.00601414 0.00595813 0.00588744]


In [47]:
print("user_item_matrix:", user_item_matrix.shape)
print("item_user_matrix:", item_user_matrix.shape)

print("ALS user factors:", als_model.user_factors.shape)
print("ALS item factors:", als_model.item_factors.shape)

print("Max item index:", len(item_ids)-1)
print("Max user index:", len(user_ids)-1)

user_item_matrix: (229536, 41194)
item_user_matrix: (41194, 229536)
ALS user factors: (41194, 64)
ALS item factors: (229536, 64)
Max item index: 41193
Max user index: 229535


In [48]:
# Test item-to-item similarity instead of recommend

similar_items, scores = als_model.similar_items(
    0,
    N=5
)

print(similar_items)
print(scores)

[ 1111 12554  1112   138     0]
[1.0000002 1.0000001 1.0000001 1.0000001 1.0000001]


In [50]:
print("Testing similar items...")

item_ids_test, scores_test = als_model.similar_items(
    itemid=0,
    N=10
)

print(item_ids_test)
print("Max returned:", max(item_ids_test))

Testing similar items...
[ 1111 65268 24273 20497 15769 13811 12554  1112   138     0]
Max returned: 65268


In [51]:
print("interaction max user_idx:",
      interaction["user_idx"].max())

print("interaction max item_idx:",
      interaction["item_idx"].max())

print("unique users:",
      len(user_ids))

print("unique items:",
      len(item_ids))

interaction max user_idx: 229535
interaction max item_idx: 41193
unique users: 229536
unique items: 41194


In [52]:
print(type(als_model))
print(als_model)

<class 'implicit.cpu.als.AlternatingLeastSquares'>


In [53]:
print("user_factors shape:", als_model.user_factors.shape)
print("item_factors shape:", als_model.item_factors.shape)

print("\nFirst few user factor rows:")
print(als_model.user_factors[:3].shape)

print("\nFirst few item factor rows:")
print(als_model.item_factors[:3].shape)

user_factors shape: (41194, 64)
item_factors shape: (229536, 64)

First few user factor rows:
(3, 64)

First few item factor rows:
(3, 64)


In [54]:
import implicit
print("implicit version:", implicit.__version__)

implicit version: 0.7.3


In [55]:
item_id_test = 0

similar_ids, scores = als_model.similar_items(
    item_id_test,
    N=10
)

print("Returned IDs:")
print(similar_ids)

print("\nAll valid item IDs?")
print(all(i < len(item_ids) for i in similar_ids))

Returned IDs:
[ 1111 65268 24273 20497 15769 13811 12554  1112   138     0]

All valid item IDs?
False


In [60]:
#als retrained and now checking 
ids, scores = get_als_recommendations(
    0,
    n=10
)

print(ids)
print("Max ID:", max(ids))

[ 4063   178  1007   582 23277   179 17823   201 22057   571]
Max ID: 23277


In [61]:
for item_id, score in zip(ids, scores):

    article_id = idx2item[item_id]

    name = article_subset[
        article_subset["article_id"] == article_id
    ]["prod_name"]

    if len(name) > 0:
        print(
            f"{name.iloc[0]} | Score: {score:.5f}"
        )

Gyda blouse | Score: 0.00459
Highwaist 30 den 1p Tights | Score: 0.00261
Greta Thong Mynta Low 3p | Score: 0.00209
Calista cardigan. | Score: 0.00195
Antonia heavy t-shirt | Score: 0.00186
Tess Tee. | Score: 0.00150
Madison skinny HW (1) | Score: 0.00115
Curvy Jeggings HW Ankle | Score: 0.00107
Sadie Shirt | Score: 0.00096
Nora T-shirt | Score: 0.00087


In [62]:
import pickle

pickle.dump(
    als_model,
    open("als_model.pkl", "wb")
)

pickle.dump(
    product_embeddings,
    open("product_embeddings.pkl", "wb")
)

pickle.dump(
    article_to_embedding_idx,
    open("article_to_embedding_idx.pkl", "wb")
)

print("All models saved successfully!")

All models saved successfully!


In [63]:
#evaluation

test_truth = test_df.groupby(
    "customer_id"
)["article_id"].apply(set).to_dict()

print("Users in test set:", len(test_truth))

Users in test set: 229536


In [64]:
#precision evaluation 
import numpy as np

sample_users = list(test_truth.keys())[:5000]

hits = 0
total = 0

for user in sample_users:

    if user not in user2idx:
        continue

    user_idx = user2idx[user]

    try:
        rec_ids, _ = als_model.recommend(
            userid=user_idx,
            user_items=user_item_matrix[user_idx],
            N=10,
            filter_already_liked_items=True
        )

        recommended_articles = {
            idx2item[item]
            for item in rec_ids
        }

        actual_articles = test_truth[user]

        if len(
            recommended_articles.intersection(
                actual_articles
            )
        ) > 0:
            hits += 1

        total += 1

    except:
        continue

precision_at_10 = hits / total

print("Evaluated users:", total)
print("Hits:", hits)
print(f"Precision@10: {precision_at_10:.4f}")

Evaluated users: 5000
Hits: 41
Precision@10: 0.0082


In [65]:
recall_at_10 = hits / total

print(f"Recall@10: {recall_at_10:.4f}")

Recall@10: 0.0082


In [66]:
import numpy as np

sample_users = list(test_truth.keys())[:5000]

ndcg_sum = 0
evaluated = 0

for user in sample_users:

    if user not in user2idx:
        continue

    user_idx = user2idx[user]

    try:

        rec_ids, _ = als_model.recommend(
            userid=user_idx,
            user_items=user_item_matrix[user_idx],
            N=10,
            filter_already_liked_items=True
        )

        recommended_articles = [
            idx2item[item]
            for item in rec_ids
        ]

        actual_item = list(test_truth[user])[0]

        if actual_item in recommended_articles:

            rank = (
                recommended_articles.index(actual_item)
                + 1
            )

            ndcg_sum += 1 / np.log2(rank + 1)

        evaluated += 1

    except:
        continue

ndcg_at_10 = ndcg_sum / evaluated

print("Users evaluated:", evaluated)
print(f"NDCG@10: {ndcg_at_10:.4f}")

Users evaluated: 5000
NDCG@10: 0.0044


In [67]:
#item embedding mapping 
item_embedding_lookup = {}

for item_idx, article_id in idx2item.items():

    if article_id in article_to_embedding_idx:

        item_embedding_lookup[item_idx] = (
            article_to_embedding_idx[article_id]
        )

print(
    "Mapped items:",
    len(item_embedding_lookup)
)

Mapped items: 41194


In [69]:
#hybrid recomedndation function
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_hybrid_recommendations(user_idx, top_n=10):

    # ALS candidates
    als_ids, als_scores = als_model.recommend(
        userid=user_idx,
        user_items=user_item_matrix[user_idx],
        N=50,
        filter_already_liked_items=True
    )

    hybrid_scores = []

    for item_id, als_score in zip(als_ids, als_scores):

        if item_id not in item_embedding_lookup:
            continue

        emb_idx = item_embedding_lookup[item_id]

        content_score = cosine_similarity(
            product_embeddings[emb_idx].reshape(1, -1),
            product_embeddings
        )[0].mean()

        final_score = (
            0.7 * float(als_score)
            + 0.3 * float(content_score)
        )

        hybrid_scores.append(
            (item_id, final_score)
        )

    hybrid_scores = sorted(
        hybrid_scores,
        key=lambda x: x[1],
        reverse=True
    )

    return hybrid_scores[:top_n]

In [70]:
#testing hybrid recomendation fun
hybrid_recs = get_hybrid_recommendations(
    0,
    top_n=10
)

for item_id, score in hybrid_recs:

    article_id = idx2item[item_id]

    name = article_subset[
        article_subset["article_id"] == article_id
    ]["prod_name"]

    if len(name) > 0:
        print(
            f"{name.iloc[0]} | {score:.4f}"
        )

Strap Top. | 0.1346
Juan | 0.1300
V-neck Strap Top. | 0.1288
Curvy Jeggings HW Ankle | 0.1267
Nora T-shirt | 0.1232
Claudine | 0.1230
Greta Thong Mynta Low 3p | 0.1203
Lucy blouse | 0.1196
CS Paula dress | 0.1194
Mona | 0.1181


In [71]:
sample_user = 0

hybrid_recs = get_hybrid_recommendations(
    sample_user,
    top_n=10
)

print("Top Hybrid Recommendations")

for item_id, score in hybrid_recs:

    article_id = idx2item[item_id]

    product = article_subset[
        article_subset["article_id"] == article_id
    ]["prod_name"]

    if len(product) > 0:
        print(product.iloc[0])

Top Hybrid Recommendations
Strap Top.
Juan
V-neck Strap Top.
Curvy Jeggings HW Ankle
Nora T-shirt
Claudine
Greta Thong Mynta Low 3p
Lucy blouse
CS Paula dress
Mona


In [72]:
sample_user = 0

user_history = user_item_matrix[sample_user].indices

print("Items purchased:", len(user_history))
print(user_history[:10])

Items purchased: 1
[0]


In [73]:
import numpy as np

user_interactions = np.diff(user_item_matrix.indptr)

print("Average interactions:", user_interactions.mean())
print("Median interactions:", np.median(user_interactions))
print("Max interactions:", user_interactions.max())

print("\nUsers with exactly 1 interaction:")
print(np.sum(user_interactions == 1))

Average interactions: 2.5034983619127282
Median interactions: 2.0
Max interactions: 65

Users with exactly 1 interaction:
102088


In [74]:
import pickle

pickle.dump(
    idx2item,
    open("idx2item.pkl", "wb")
)

pickle.dump(
    item_embedding_lookup,
    open("item_embedding_lookup.pkl", "wb")
)

pickle.dump(
    article_subset,
    open("article_subset.pkl", "wb")
)

print("Metadata saved!")

Metadata saved!


In [75]:
print(article_subset.columns.tolist())

['article_id', 'prod_name', 'product_type_name', 'product_group_name', 'colour_group_name', 'detail_desc', 'text']


In [76]:
article_subset["product_type_name"].value_counts().head(20)

product_type_name
Dress               4820
Trousers            4375
Sweater             3579
T-shirt             2533
Top                 1822
Blouse              1810
Shorts              1522
Shirt               1351
Skirt               1347
Jacket              1334
Vest top            1333
Bra                 1136
Underwear bottom    1125
Hoodie               785
Socks                678
Swimwear bottom      665
Leggings/Tights      659
Bikini top           596
Cardigan             549
Blazer               487
Name: count, dtype: int64

In [79]:
#category-based product sampler
def get_products_by_category(category, n=10):

    products = article_subset[
        article_subset["product_type_name"] == category
    ][["article_id", "prod_name"]]

    return products.sample(
        min(n, len(products)),
        random_state=42
    )

In [80]:
sample_products = get_products_by_category(
    "T-shirt",
    n=10
)

sample_products

,article_id,prod_name
97404,0874122001,Price TEE - TVP
90090,0841402004,Grimsby
15497,0567418047,MAXENCE TEE
74413,0776781002,D2 PE LARA WOOL JERSEY
49857,0695457004,Lillibeth jumper
12789,0554772002,Bob V-neck
81143,0804741005,Pria tee
98650,0880238003,Daisy tee
62703,0737040007,ESSENTIAL SIGNE BOAT NECK
68121,0753255002,Terrier- TVP- TM


In [81]:
#product name search
product_name_to_article = dict(
    zip(
        article_subset["prod_name"],
        article_subset["article_id"]
    )
)

print(
    "Products mapped:",
    len(product_name_to_article)
)

Products mapped: 20757


In [82]:
print("embedding_idx_to_article" in globals())

False


In [83]:
#creating reverse mapping 
embedding_idx_to_article = {
    v: k
    for k, v in article_to_embedding_idx.items()
}

print(
    "Embeddings mapped:",
    len(embedding_idx_to_article)
)

Embeddings mapped: 41194


In [96]:
#cold start rec fn
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_from_likes(
    liked_product_names,
    top_n=10
):

    liked_embeddings = []

    for product_name in liked_product_names:

        row = article_subset[
            article_subset["prod_name"] == product_name
        ]

        if len(row) == 0:
            continue

        article_id = row.iloc[0]["article_id"]

        if article_id not in article_to_embedding_idx:
            continue

        emb_idx = article_to_embedding_idx[
            article_id
        ]

        liked_embeddings.append(
            product_embeddings[emb_idx]
        )

    if len(liked_embeddings) == 0:
        return []

    user_profile = np.mean(
        liked_embeddings,
        axis=0
    )

    similarities = cosine_similarity(
        user_profile.reshape(1, -1),
        product_embeddings
    )[0]

    ranked_indices = similarities.argsort()[::-1]

    recommendations = []

    liked_set = set(liked_product_names)

    seen_products = set()

    for idx in ranked_indices:

        article_id = embedding_idx_to_article[idx]

        row = article_subset[
            article_subset["article_id"] == article_id
        ]

        if len(row) == 0:
            continue

        name = row.iloc[0]["prod_name"]

        if name in liked_set:
            continue

        if name in seen_products:
            continue

        seen_products.add(name)

        recommendations.append(
            (name, similarities[idx])
        )

        if len(recommendations) >= top_n:
            break

    return recommendations

In [97]:
#test
liked_products = [
    "Price TEE - TVP",
    "Pria tee",
    "Daisy tee"
]

recommendations = recommend_from_likes(
    liked_products,
    top_n=10
)

for name, score in recommendations:
    print(
        f"{name} | {score:.4f}"
    )

Pretty Tee | 0.8603
PRINTED TEE | 0.8583
Utility Tee | 0.8528
daisy tee | 0.8520
Shoulder tee | 0.8514
Tory price tee | 0.8492
Taz Tee | 0.8478
Rich Tee | 0.8468
Coin tee | 0.8455
LOW PRICE TEE | 0.8432


In [98]:
import pickle

pickle.dump(
    als_model,
    open("als_model.pkl", "wb")
)

pickle.dump(
    product_embeddings,
    open("product_embeddings.pkl", "wb")
)

pickle.dump(
    article_to_embedding_idx,
    open("article_to_embedding_idx.pkl", "wb")
)

pickle.dump(
    embedding_idx_to_article,
    open("embedding_idx_to_article.pkl", "wb")
)

pickle.dump(
    article_subset,
    open("article_subset.pkl", "wb")
)

print("All project files saved!")

All project files saved!
